In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

# Load the GTD dataset
df = pd.read_csv("../data/raw/globalterrorismdb_1970_2020.csv")
df.head(5)

C:\Users\USER\AppData\Local\Temp\ipykernel_9172\100574391.py:4: DtypeWarning: Columns (4,31,33,54,61,62,63,76,79,90,92,94,96,114,115,121) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/globalterrorismdb_1970_2020.csv")


,eventid,iyear,imonth,iday,approxdate,extended,resolution,country,country_txt,region,...,addnotes,scite1,scite2,scite3,dbsource,INT_LOG,INT_IDEO,INT_MISC,INT_ANY,related
0,197000000001,1970,7,2,NaN,0,NaN,58,Dominican Republic,2,...,NaN,NaN,NaN,NaN,PGIS,0,0,0,0,NaN
1,197000000002,1970,0,0,NaN,0,NaN,130,Mexico,1,...,NaN,NaN,NaN,NaN,PGIS,0,1,1,1,NaN
2,197001000001,1970,1,0,NaN,0,NaN,160,Philippines,5,...,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN
3,197001000002,1970,1,0,NaN,0,NaN,78,Greece,8,...,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN
4,197001000003,1970,1,0,NaN,0,NaN,101,Japan,4,...,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN


In [ ]:
# Load columns of interest
columns_of_interest = [
    "eventid",
    "iyear",
    "imonth",
    "iday",
    "country_txt",
    "region_txt",
    "provstate",
    "city",
    "latitude",
    "longitude",
    "attacktype1_txt",
    "targtype1_txt",
    "weaptype1_txt",
    "gname",
    "nkill",
    "nwound",
    "success",
    "suicide",
    "summary"
]

df = pd.read_csv(
    "../data/raw/globalterrorismdb_1970_2020.csv",
    usecols=columns_of_interest,
    low_memory=False
)

In [5]:
df.shape

(209706, 19)

In [6]:
df.isnull().sum()

eventid                0
iyear                  0
imonth                 0
iday                   0
country_txt            0
region_txt             0
provstate              0
city                 427
latitude            4691
longitude           4692
summary            66120
success                0
suicide                0
attacktype1_txt        0
targtype1_txt          0
gname                  0
weaptype1_txt          0
nkill              12527
nwound             19936
dtype: int64

In [ ]:
# Create a copy of the dataset for cleaning
terror_df = df.copy()

In [ ]:
# Replace missing city names with "Unknown"
terror_df["city"] = terror_df["city"].fillna("Unknown")

# Replace missing fatalities and injuries with zero
terror_df["nkill"] = terror_df["nkill"].fillna(0)
terror_df["nwound"] = terror_df["nwound"].fillna(0)

In [11]:
terror_df.dtypes

eventid              int64
iyear                int64
imonth               int64
iday                 int64
country_txt         object
region_txt          object
provstate           object
city                object
latitude           float64
longitude          float64
summary             object
success              int64
suicide              int64
attacktype1_txt     object
targtype1_txt       object
gname               object
weaptype1_txt       object
nkill              float64
nwound             float64
dtype: object

In [ ]:
# Replace "0" month and day values with 1
terror_df["imonth"] = terror_df["imonth"].replace(0,1)

terror_df["iday"] = terror_df["iday"].replace(0,1)

In [ ]:
# Create a date column by combining year, month, and day
terror_df["date"] = pd.to_datetime(
    terror_df["iyear"].astype(str)
    + "-"
    + terror_df["imonth"].astype(str)
    + "-"
    + terror_df["iday"].astype(str)
)

terror_df["date"].head(5)

0   1970-07-02
1   1970-01-01
2   1970-01-01
3   1970-01-01
4   1970-01-01
Name: date, dtype: datetime64[ns]

In [ ]:
# Create total casualties variable combining fatalities and injuries
terror_df["casualties"] = (
    terror_df["nkill"] +
    terror_df["nwound"]
)

terror_df["casualties"].head(5)

0    1.0
1    0.0
2    1.0
3    0.0
4    0.0
Name: casualties, dtype: float64

In [ ]:
# Standardize unknown group names
terror_df["gname"] = terror_df["gname"].replace(
    "Unknown",
    "Unknown Group"
)

terror_df["gname"].head(5)

0                                MANO-D
1    23rd of September Communist League
2                         Unknown Group
3                         Unknown Group
4                         Unknown Group
Name: gname, dtype: object

In [ ]:
# Label attacks as successful or failed
terror_df["attack_success"] = terror_df["success"].map(
    {
        1:"Successful",
        0:"Failed"
    }
)

terror_df["attack_success"].value_counts()

attack_success
Successful    185302
Failed         24404
Name: count, dtype: int64

In [ ]:
# Label suicide attacks
terror_df["suicide_attack"] = terror_df["suicide"].map(
    {
        1:"Yes",
        0:"No"
    }
)

terror_df["suicide_attack"].value_counts()

suicide_attack
No     202268
Yes      7438
Name: count, dtype: int64

In [ ]:
# Create decade variable
terror_df["decade"] = (
    terror_df["iyear"] // 10
) * 10

terror_df["decade"] = (
    terror_df["decade"]
    .astype(str)
    + "s"
)

terror_df["decade"].value_counts()

decade
2010s    106377
1980s     31156
1990s     28765
2000s     25057
1970s      9913
2020s      8438
Name: count, dtype: int64

In [ ]:
# Identify high-casualty incidents
terror_df["high_casualty"] = np.where(
    terror_df["casualties"] >= 50,
    "Yes",
    "No"
)

terror_df["high_casualty"].value_counts()

high_casualty
No     206759
Yes      2947
Name: count, dtype: int64

In [23]:
terror_df.head(5)

,eventid,iyear,imonth,iday,country_txt,region_txt,provstate,city,latitude,longitude,...,gname,weaptype1_txt,nkill,nwound,date,casualties,attack_success,suicide_attack,decade,high_casualty
0,197000000001,1970,7,2,Dominican Republic,Central America & Caribbean,National,Santo Domingo,18.456792,-69.951164,...,MANO-D,Unknown,1.0,0.0,1970-07-02,1.0,Successful,No,1970s,No
1,197000000002,1970,1,1,Mexico,North America,Federal,Mexico city,19.371887,-99.086624,...,23rd of September Communist League,Unknown,0.0,0.0,1970-01-01,0.0,Successful,No,1970s,No
2,197001000001,1970,1,1,Philippines,Southeast Asia,Tarlac,Unknown,15.478598,120.599741,...,Unknown Group,Unknown,1.0,0.0,1970-01-01,1.0,Successful,No,1970s,No
3,197001000002,1970,1,1,Greece,Western Europe,Attica,Athens,37.997490,23.762728,...,Unknown Group,Explosives,0.0,0.0,1970-01-01,0.0,Successful,No,1970s,No
4,197001000003,1970,1,1,Japan,East Asia,Fukouka,Fukouka,33.580412,130.396361,...,Unknown Group,Incendiary,0.0,0.0,1970-01-01,0.0,Successful,No,1970s,No


In [27]:
terror_df["summary"].isna().sum()

np.int64(66120)

In [28]:
terror_df["summary"] = terror_df["summary"].fillna(
    "No summary available"
)

In [29]:
terror_df["summary"].isna().sum()

np.int64(0)

In [30]:
terror_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209706 entries, 0 to 209705
Data columns (total 25 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   eventid          209706 non-null  int64         
 1   iyear            209706 non-null  int64         
 2   imonth           209706 non-null  int64         
 3   iday             209706 non-null  int64         
 4   country_txt      209706 non-null  object        
 5   region_txt       209706 non-null  object        
 6   provstate        209706 non-null  object        
 7   city             209706 non-null  object        
 8   latitude         205015 non-null  float64       
 9   longitude        205014 non-null  float64       
 10  summary          209706 non-null  object        
 11  success          209706 non-null  int64         
 12  suicide          209706 non-null  int64         
 13  attacktype1_txt  209706 non-null  object        
 14  targtype1_txt    209

In [25]:
df.describe()

,eventid,iyear,imonth,iday,latitude,longitude,success,suicide,nkill,nwound
count,2.097060e+05,209706.000000,209706.000000,209706.000000,205015.000000,205014.000000,209706.000000,209706.000000,197179.000000,189770.000000
mean,2.004867e+11,2004.800993,6.455285,15.527930,23.358696,30.416738,0.883628,0.035469,2.431030,3.085872
std,1.351933e+09,13.519321,3.387098,8.801104,18.137061,56.113029,0.320672,0.184962,11.340882,40.916175
min,1.970000e+11,1970.000000,0.000000,0.000000,-53.154613,-176.176447,0.000000,0.000000,0.000000,0.000000
25%,1.992080e+11,1992.000000,4.000000,8.000000,11.510046,8.748117,1.000000,0.000000,0.000000,0.000000
50%,2.012010e+11,2012.000000,6.000000,15.000000,31.300213,43.746215,1.000000,0.000000,0.000000,0.000000
75%,2.015123e+11,2015.000000,9.000000,23.000000,34.557022,68.835918,1.000000,0.000000,2.000000,2.000000
max,2.020123e+11,2020.000000,12.000000,31.000000,74.633553,179.366667,1.000000,1.000000,1700.000000,10878.000000


In [31]:
# Save cleaned dataset
terror_df.to_csv(
    "../data/processed/terrorism_cleaned.csv",
    index=False
)